<a href="https://colab.research.google.com/github/diaoumardia2001-beep/DI-Bootcamp-May/blob/main/Exercises_XP_Day4_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [1]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [2]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [4]:
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # 1. Project input from in_dim down to rank dimension: (batch, in_dim) @ (in_dim, rank) -> (batch, rank)
        # 2. Project from rank dimension up to out_dim: (batch, rank) @ (rank, out_dim) -> (batch, out_dim)
        # 3. Scale by alpha
        lora_update = (x @ self.A) @ self.B
        x = self.alpha * lora_update
        return x

# Hyperparameters for the sandbox test
random_seed = 123
in_dim = 4      # Number of input features
out_dim = 3     # Number of output features
rank = 2        # Low-rank dimension (bottleneck)
alpha = 1       # Scaling factor

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)

# Sample input with batch size of 2
x = torch.randn(2, in_dim)

print("Input Tensor x:\n", x)
print("\nLayer details:\n", layer)
print("\nOriginal output:\n", layer(x))

Input Tensor x:
 tensor([[ 0.3239, -0.1085,  0.2103, -0.3908],
        [ 0.2350,  0.6653,  0.3528,  0.9728]])

Layer details:
 LoRALayer()

Original output:
 tensor([[0., 0., 0.],
        [0., 0., 0.]], grad_fn=<MulBackward0>)


# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [10]:
import torch
import torch.nn as nn

class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # Math: Output = Base Pre-trained Output + Scaled Low-Rank Adaptation
        return self.linear(x) + self.lora(x)

# Re-using the dimensions from your sandbox variables (in_dim=4, out_dim=3)
base_linear = nn.Linear(4, 3)

# Wrap the standard linear layer with our custom LoRA adaptation pathway
layer_lora_1 = LinearWithLoRA(base_linear, rank=2, alpha=1)

print("LinearWithLoRA output:\n", layer_lora_1(x))

LinearWithLoRA output:
 tensor([[ 0.3509,  0.3081,  0.2465],
        [ 0.6483, -0.4089,  0.0543]], grad_fn=<AddBackward0>)


# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [8]:
import torch
import torch.nn as nn

class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

# 1. Define input parameters matching our dimensions
num_features = 4
num_classes = 3

single_net = SingleLayerNet(num_features=num_features, num_classes=num_classes)
sample_input = torch.randn(2, num_features)  # Batch size of 2

# 2. Get the clean pre-trained baseline output
with torch.no_grad():
    baseline_output = single_net(sample_input)

# 3. Swap the standard Linear layer with our LoRA-enhanced counterpart
# We extract the existing layer directly, keeping its exact weights intact
single_net.layer = LinearWithLoRA(single_net.layer, rank=2, alpha=1)

# 4. Extract the newly wrapped output prediction matrix
with torch.no_grad():
    lora_output = single_net(sample_input)

# 5. Check if they are strictly equal (using torch.allclose to catch small float precision noise)
outputs_match = torch.allclose(baseline_output, lora_output)

print("Baseline Output:\n", baseline_output)
print("\nLoRA Output:\n", lora_output)
print("\nOutputs match before training?", outputs_match)

Baseline Output:
 tensor([[-1.1301,  0.6608, -0.8637],
        [-0.5266,  0.2062, -0.3426]])

LoRA Output:
 tensor([[-1.1301,  0.6608, -0.8637],
        [-0.5266,  0.2062, -0.3426]])

Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [11]:
import torch
import torch.nn.functional as F

class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # 1. Combine LoRA matrices into a single update matrix: (in_features, rank) @ (rank, out_features) -> (in_features, out_features)
        lora = self.lora.A @ self.lora.B

        # 2. Merge original weights with the scaled LoRA weights.
        # Note: PyTorch's F.linear expects the weight matrix to be shaped as (out_features, in_features).
        # Since our `lora` matrix is computed as (in_features, out_features), we must transpose it using .T before adding it.
        combined_weight = self.linear.weight + self.lora.alpha * lora.T

        return F.linear(x, combined_weight, self.linear.bias)

# 3. Instantiate the merged layer using your existing `base_linear` instance, a rank of 2, and an alpha of 1
layer_lora_2 = LinearWithLoRAMerged(base_linear, rank=2, alpha=1)

print("Merged LoRA output:\n", layer_lora_2(x))

Merged LoRA output:
 tensor([[ 0.3509,  0.3081,  0.2465],
        [ 0.6483, -0.4089,  0.0543]], grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [13]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            # Flatten the input images from (batch, 1, 28, 28) down to (batch, 784)
            nn.Flatten(),

            # Layer 1: Input features -> First hidden layer
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),

            # Layer 2: First hidden layer -> Second hidden layer
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),

            # Layer 3: Second hidden layer -> Output classes
            nn.Linear(num_hidden_2, num_classes),
        )

    def forward(self, x):
        x = self.layers(x)
        return x

In [15]:
# Architecture
num_features = 28 * 28    # MNIST images are 28x28 pixels = 784 input features
num_hidden_1 = 128        # Dimensions for the first hidden layer
num_hidden_2 = 64         # Dimensions for the second hidden layer
num_classes = 10          # MNIST has 10 digit classes (0 through 9)

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 0.005     # A stable baseline learning rate for the Adam optimizer
num_epochs = 2            # Kept short (2 epochs) for fast validation in your notebook challenge

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)

# Initialize the Adam optimizer with the pre-trained model's parameters and learning rate
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Active Device:", DEVICE)
print("\n--- Model Architecture ---")
print(model)
print("\n--- Optimizer Configuration ---")
print(optimizer_pretrained)

Active Device: cpu

--- Model Architecture ---
MultilayerPerceptron(
  (layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=10, bias=True)
  )
)

--- Optimizer Configuration ---
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [17]:
BATCH_SIZE = 64

# Note: transforms.ToTensor() scales input images to 0-1 range
train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)

# 1. Instantiate the test partition (train=False)
test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

# 2. Build the training data loader (shuffle=True to ensure diverse batches during training)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 3. Build the testing data loader (shuffle=False since evaluation order doesn't affect tracking)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Verification Loop
for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Define evaluation

In [18]:
def compute_accuracy(model, data_loader, device):
    model.eval()  # Switch model to evaluation mode (deactivates dropout/batchnorm)
    correct_pred, num_examples = 0, 0

    with torch.no_grad():  # Deactivate gradient tracking to save memory and compute
        for features, targets in data_loader:
            # 1. Ship your data tensors over to your hardware target (CPU or GPU)
            features = features.to(device)
            targets = targets.to(device)

            # 2. Extract model raw predictions (logits) via a forward pass
            logits = model(features)

            # Extract predicted index matching the highest logit score along the class dimension (dim=1)
            _, predicted_labels = torch.max(logits, 1)

            # 3. Accumulate total sample size count and total correct matching instances
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum().item()

    # 4. Compute percentage scale: (Correct Predictions / Total Samples) * 100
    return (correct_pred / num_examples) * 100

## Training

In [19]:
import time
import torch.nn.functional as F

def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()  # Set the model to training mode
        for batch_idx, (features, targets) in enumerate(train_loader):
            # 1. Move input data and targets to the device (CPU or GPU)
            features = features.to(device)
            targets = targets.to(device)

            # 2. Forward pass to compute raw predictions (logits)
            logits = model(features)

            # 3. Compute cross-entropy loss between predictions and actual labels
            loss = F.cross_entropy(logits, targets)

            # 4. Clear existing gradients from the previous step
            optimizer.zero_grad()

            # 5. Backward pass to compute new gradients
            loss.backward()

            # 6. Update the model parameters using the optimizer
            optimizer.step()

            # Logging metrics every 400 batches
            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        # Evaluate training accuracy at the end of each epoch
        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [20]:
# Execute the training run using the pre-trained optimizer
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)

# Evaluate and display the final baseline test accuracy
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

/tmp/ipykernel_2564/2641248598.py:30: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))


Epoch: 001/002|Batch 000/938| Loss: 2.3026
Epoch: 001/002|Batch 400/938| Loss: 0.1462
Epoch: 001/002|Batch 800/938| Loss: 0.2556
Epoch: 001/002 training accuracy: 96.62%
Time elapsed: 0.28 min
Epoch: 002/002|Batch 000/938| Loss: 0.0306
Epoch: 002/002|Batch 400/938| Loss: 0.1255
Epoch: 002/002|Batch 800/938| Loss: 0.1672
Epoch: 002/002 training accuracy: 97.32%
Time elapsed: 0.57 min
Total Training Time: 0.57 min
Test accuracy: 96.61%


# Replacing Linear with LoRA Layers

In [21]:
import copy

# Create a deep copy of the pre-trained base model
model_lora = copy.deepcopy(model)

# Replace the original standard nn.Linear layers with our LoRA-enhanced variants
model_lora.layers[1] = LinearWithLoRAMerged(model_lora.layers[1], rank=4, alpha=8)
model_lora.layers[3] = LinearWithLoRAMerged(model_lora.layers[3], rank=4, alpha=8)
model_lora.layers[5] = LinearWithLoRAMerged(model_lora.layers[5], rank=4, alpha=8)

# Ship the newly constructed architecture back to your active hardware engine
model_lora.to(DEVICE)

# Set up a fresh optimizer for tracking parameter gradients
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)

print("--- Modified LoRA Network Architecture ---")
print(model_lora)

print(f'\nTest accuracy orig model: {compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

--- Modified LoRA Network Architecture ---
MultilayerPerceptron(
  (layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (2): ReLU()
    (3): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=64, bias=True)
      (lora): LoRALayer()
    )
    (4): ReLU()
    (5): LinearWithLoRAMerged(
      (linear): Linear(in_features=64, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)

Test accuracy orig model: 96.61%
Test accuracy LoRA model: 96.61%


## Freezing the Original Linear Layers

In [22]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}:{param.requires_grad}')

layers.1.linear.weight:False
layers.1.linear.bias:False
layers.1.lora.A:True
layers.1.lora.B:True
layers.3.linear.weight:False
layers.3.linear.bias:False
layers.3.lora.A:True
layers.3.lora.B:True
layers.5.linear.weight:False
layers.5.linear.bias:False
layers.5.lora.A:True
layers.5.lora.B:True


In [23]:
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

Epoch: 001/002|Batch 000/938| Loss: 0.0474
Epoch: 001/002|Batch 400/938| Loss: 0.2108
Epoch: 001/002|Batch 800/938| Loss: 0.0218
Epoch: 001/002 training accuracy: 97.80%
Time elapsed: 0.30 min
Epoch: 002/002|Batch 000/938| Loss: 0.0071
Epoch: 002/002|Batch 400/938| Loss: 0.0197
Epoch: 002/002|Batch 800/938| Loss: 0.0503
Epoch: 002/002 training accuracy: 98.03%
Time elapsed: 0.58 min
Total Training Time: 0.58 min
Test accuracy LoRA finetune: 97.14%
Test accuracy orig model:96.61%
Test accuracy LoRA model:97.14%
